In [2]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
BUCKET = "gs://retailsmart-lake-retailsmart-bd"

spark.conf.set("spark.sql.catalog.retailsmart", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.retailsmart.type", "hadoop")
spark.conf.set("spark.sql.catalog.retailsmart.warehouse", f"{BUCKET}/lakehouse")
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.sql("CREATE NAMESPACE IF NOT EXISTS retailsmart.universal")

DataFrame[]

In [3]:
tablas = ["users", "orders", "order_items", "products",
          "inventory_items", "distribution_centers", "events"]
for t in tablas:
    d = spark.read.parquet(f"{BUCKET}/landing/thelook/{t}/")
    print(f"== {t}: {d.count()} filas")
    d.printSchema()

== users: 100000 filas
root
 |-- id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- state: string (nullable = true)
 |-- street_address: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- traffic_source: string (nullable = true)
 |-- created_at: timestamp_ntz (nullable = true)
 |-- user_geom: binary (nullable = true)



== orders: 125138 filas


root
 |-- order_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- created_at: timestamp_ntz (nullable = true)
 |-- returned_at: timestamp_ntz (nullable = true)
 |-- shipped_at: timestamp_ntz (nullable = true)
 |-- delivered_at: timestamp_ntz (nullable = true)
 |-- num_of_item: long (nullable = true)



== order_items: 181644 filas


root
 |-- id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- inventory_item_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: timestamp_ntz (nullable = true)
 |-- shipped_at: timestamp_ntz (nullable = true)
 |-- delivered_at: timestamp_ntz (nullable = true)
 |-- returned_at: timestamp_ntz (nullable = true)
 |-- sale_price: double (nullable = true)



...

== inventory_items: 490408 filas


root
 |-- id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- created_at: timestamp_ntz (nullable = true)
 |-- sold_at: timestamp_ntz (nullable = true)
 |-- cost: double (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_retail_price: double (nullable = true)
 |-- product_department: string (nullable = true)
 |-- product_sku: string (nullable = true)
 |-- product_distribution_center_id: long (nullable = true)



== distribution_centers: 10 filas
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- distribution_center_geom: binary (nullable = true)



== events: 2429664 filas


root
 |-- id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- sequence_number: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- created_at: timestamp_ntz (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- browser: string (nullable = true)
 |-- traffic_source: string (nullable = true)
 |-- uri: string (nullable = true)
 |-- event_type: string (nullable = true)



In [4]:
col = F.col

def normaliza(df):
    for f in df.schema.fields:
        if f.dataType.simpleString() == "timestamp_ntz":
            df = df.withColumn(f.name, F.col(f.name).cast("timestamp"))
    return df

def thelook_a_universal(tabla, entidad, clave, reglas, quitar=(), dominio="ecommerce"):
    raw = normaliza(spark.read.parquet(f"{BUCKET}/landing/thelook/{tabla}/"))
    if quitar:
        raw = raw.drop(*quitar)
    checked = raw.withColumn("motivo", F.coalesce(*[F.when(c, F.lit(t)) for t, c in reglas]))
    malos = checked.filter(col("motivo").isNotNull())
    buenos = checked.filter(col("motivo").isNull()).drop("motivo").dropDuplicates(clave)
    meta = lambda d: (d.withColumn("_fuente", F.lit("thelook"))
                       .withColumn("_cargado_en", F.current_timestamp()))
    t_ok = f"retailsmart.universal.silver_{dominio}_{entidad}_v1"
    t_bad = f"retailsmart.universal.cuarentena_{dominio}_{entidad}_v1"
    meta(buenos).writeTo(t_ok).using("iceberg").createOrReplace()
    meta(malos).writeTo(t_bad).using("iceberg").createOrReplace()
    n_raw = raw.count()
    n_ok = spark.table(t_ok).count()
    n_bad = spark.table(t_bad).count()
    print(f"{entidad}: leídas {n_raw} | válidas {n_ok} | rechazadas {n_bad} | duplicados {n_raw - n_ok - n_bad}")

In [5]:
thelook_a_universal("users", "usuarios", ["id"], [
    ("id nulo", col("id").isNull()),
    ("fecha de alta nula", col("created_at").isNull()),
    ("edad fuera de rango", (col("age") < 0) | (col("age") > 120)),
    ("coordenada inválida", ~col("latitude").between(-90, 90) | ~col("longitude").between(-180, 180))],
    quitar=["user_geom"])

thelook_a_universal("orders", "pedidos", ["order_id"], [
    ("order_id nulo", col("order_id").isNull()),
    ("user_id nulo", col("user_id").isNull()),
    ("fecha de creación nula", col("created_at").isNull()),
    ("cantidad de ítems menor a 1", col("num_of_item") < 1),
    ("fechas fuera de secuencia",
     (col("shipped_at") < col("created_at")) | (col("delivered_at") < col("shipped_at"))
     | (col("returned_at") < col("delivered_at")))])

thelook_a_universal("order_items", "items_pedido", ["id"], [
    ("id nulo", col("id").isNull()),
    ("order_id nulo", col("order_id").isNull()),
    ("product_id nulo", col("product_id").isNull()),
    ("inventory_item_id nulo", col("inventory_item_id").isNull()),
    ("precio de venta inválido", col("sale_price").isNull() | (col("sale_price") < 0)),
    ("fechas fuera de secuencia",
     (col("shipped_at") < col("created_at")) | (col("delivered_at") < col("shipped_at"))
     | (col("returned_at") < col("delivered_at")))])

thelook_a_universal("products", "productos", ["id"], [
    ("id nulo", col("id").isNull()),
    ("costo o precio negativo", (col("cost") < 0) | (col("retail_price") < 0))])

thelook_a_universal("inventory_items", "inventario", ["id"], [
    ("id nulo", col("id").isNull()),
    ("product_id nulo", col("product_id").isNull()),
    ("costo negativo", col("cost") < 0),
    ("vendido antes de crearse", col("sold_at") < col("created_at"))])

thelook_a_universal("distribution_centers", "centros_distribucion", ["id"], [
    ("id nulo", col("id").isNull()),
    ("coordenada inválida", ~col("latitude").between(-90, 90) | ~col("longitude").between(-180, 180))],
    quitar=["distribution_center_geom"])

thelook_a_universal("events", "eventos", ["id"], [
    ("id nulo", col("id").isNull()),
    ("session_id nulo", col("session_id").isNull()),
    ("fecha nula", col("created_at").isNull()),
    ("tipo de evento nulo", col("event_type").isNull())])

26/10/18 03:44:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


usuarios: leídas 100000 | válidas 100000 | rechazadas 0 | duplicados 0


pedidos: leídas 125138 | válidas 125138 | rechazadas 0 | duplicados 0


items_pedido: leídas 181644 | válidas 145978 | rechazadas 35666 | duplicados 0


productos: leídas 29120 | válidas 29120 | rechazadas 0 | duplicados 0


inventario: leídas 490408 | válidas 490408 | rechazadas 0 | duplicados 0


centros_distribucion: leídas 10 | válidas 10 | rechazadas 0 | duplicados 0


eventos: leídas 2429664 | válidas 2429664 | rechazadas 0 | duplicados 0


In [6]:
q = spark.table("retailsmart.universal.cuarentena_ecommerce_items_pedido_v1")
q.groupBy("motivo").count().show(truncate=False)
q.groupBy("motivo", "status").count().orderBy(F.desc("count")).show(truncate=False)
q.select("id", "order_id", "status", "created_at", "shipped_at", "delivered_at",
         "returned_at", "sale_price", "inventory_item_id", "motivo").show(5, truncate=False)

+-------------------------+-----+
|motivo                   |count|
+-------------------------+-----+
|fechas fuera de secuencia|35666|
+-------------------------+-----+



+-------------------------+--------+-----+
|motivo                   |status  |count|
+-------------------------+--------+-----+
|fechas fuera de secuencia|Shipped |16664|
|fechas fuera de secuencia|Complete|13720|
|fechas fuera de secuencia|Returned|5282 |
+-------------------------+--------+-----+



+------+--------+--------+-------------------+-------------------+-------------------+-----------+------------------+-----------------+-------------------------+
|id    |order_id|status  |created_at         |shipped_at         |delivered_at       |returned_at|sale_price        |inventory_item_id|motivo                   |
+------+--------+--------+-------------------+-------------------+-------------------+-----------+------------------+-----------------+-------------------------+
|133234|91904   |Complete|2022-07-07 12:10:29|2022-07-06 08:41:08|2022-07-08 12:03:08|NULL       |0.0199999995529651|359685           |fechas fuera de secuencia|
|57230 |39492   |Complete|2025-04-04 06:04:06|2025-04-04 01:32:13|2025-04-05 23:51:13|NULL       |0.4900000095367431|154274           |fechas fuera de secuencia|
|176707|121742  |Complete|2022-10-11 16:18:39|2022-10-11 08:38:29|2022-10-11 19:39:29|NULL       |0.4900000095367431|477081           |fechas fuera de secuencia|
|75340 |52025   |Shipped |20

In [7]:
raw = normaliza(spark.read.parquet(f"{BUCKET}/landing/thelook/order_items/"))

reglas = [
    ("id nulo", col("id").isNull()),
    ("order_id nulo", col("order_id").isNull()),
    ("product_id nulo", col("product_id").isNull()),
    ("inventory_item_id nulo", col("inventory_item_id").isNull()),
    ("precio de venta inválido", col("sale_price").isNull() | (col("sale_price") < 0)),
]
checked = raw.withColumn("motivo", F.coalesce(*[F.when(c, F.lit(t)) for t, c in reglas]))
malos = checked.filter(col("motivo").isNotNull())
incoherente = F.coalesce(
    (col("shipped_at") < col("created_at")) | (col("delivered_at") < col("shipped_at"))
    | (col("returned_at") < col("delivered_at")), F.lit(False))
buenos = (checked.filter(col("motivo").isNull()).drop("motivo")
          .dropDuplicates(["id"]).withColumn("fechas_inconsistentes", incoherente))

meta = lambda d: (d.withColumn("_fuente", F.lit("thelook"))
                   .withColumn("_cargado_en", F.current_timestamp()))
t_ok = "retailsmart.universal.silver_ecommerce_items_pedido_v1"
t_bad = "retailsmart.universal.cuarentena_ecommerce_items_pedido_v1"
meta(buenos).writeTo(t_ok).using("iceberg").createOrReplace()
meta(malos).writeTo(t_bad).using("iceberg").createOrReplace()

print("leídas:", raw.count(), "| válidas:", spark.table(t_ok).count(), "| rechazadas:", spark.table(t_bad).count())
spark.table(t_ok).groupBy("fechas_inconsistentes").count().show()

leídas: 181644 | válidas: 181644 | rechazadas: 0


+---------------------+------+
|fechas_inconsistentes| count|
+---------------------+------+
|                 true| 35666|
|                false|145978|
+---------------------+------+

